# 本章总结

本章主要学习 PyTorch 中如何使用 CPU 和 GPU 进行计算。核心思想是：**张量和模型参数都存放在某个设备上，参与同一次计算的对象必须位于同一个设备**。

主要内容包括：

- **确认 GPU 状态**：使用 `nvidia-smi` 查看机器是否有 NVIDIA GPU，以及 GPU 的显存占用和运行状态。
- **认识设备对象**：PyTorch 使用 `torch.device('cpu')`、`torch.device('cuda')`、`torch.device('cuda:0')` 等方式表示 CPU 或 GPU。
- **检查 GPU 数量**：通过 `torch.cuda.device_count()` 获取当前可用 GPU 的数量。
- **编写设备选择函数**：`try_gpu()` 用于在 GPU 可用时返回指定 GPU，否则自动退回 CPU；`try_all_gpus()` 用于返回所有可用 GPU。
- **查看张量所在设备**：通过 `X.device` 可以确认张量当前存储在 CPU 还是 GPU 上。
- **在指定设备创建张量**：创建张量时可以传入 `device=try_gpu()`，让张量直接生成在目标设备上。
- **移动张量到设备**：可以使用 `.to(device)` 或 `.cuda(i)` 将张量移动到指定设备；初学者更推荐 `.to(device)`，因为它可以同时兼容 CPU 和 GPU。
- **同设备计算原则**：执行 `X + Y`、矩阵乘法、模型前向传播等操作时，相关张量必须在同一个设备上，否则会报错。
- **模型迁移到 GPU**：使用 `net.to(device)` 可以把神经网络中的所有参数移动到指定设备。
- **输入和模型保持一致**：如果模型在 GPU 上，输入张量也必须在同一个 GPU 上；如果模型在 CPU 上，输入也应在 CPU 上。

本章最重要的实践经验是：先统一定义设备，例如 `device = try_gpu()`，然后让数据和模型都使用同一个 `device`。这样代码既能在有 GPU 的机器上加速运行，也能在没有 GPU 的机器上自动退回 CPU。

In [4]:
# ===== 本章综合代码示例：让张量和模型自动使用 GPU，没有 GPU 就使用 CPU =====
# 这个示例把本章知识点串起来：
# 1. 检查 GPU 是否可用；
# 2. 选择计算设备；
# 3. 把张量放到同一个设备；
# 4. 把神经网络模型放到同一个设备；
# 5. 在该设备上完成一次简单训练；
# 6. 检查模型参数所在设备。

import torch
from torch import nn

# ------------------------------------------------------------
# 1. 定义设备选择函数
# ------------------------------------------------------------

def try_gpu(i=0):
    """如果第 i 块 GPU 存在，就返回 cuda:i；否则返回 cpu。"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')


def try_all_gpus():
    """返回所有可用 GPU；如果没有 GPU，就返回 [cpu]。"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]
    return devices if devices else [torch.device('cpu')]

# 统一选择一个设备。后面的张量和模型都放到这个 device 上。
device = try_gpu()

print('GPU 数量:', torch.cuda.device_count())
print('所有可用设备:', try_all_gpus())
print('当前使用设备:', device)

# ------------------------------------------------------------
# 2. 张量默认在 CPU 上
# ------------------------------------------------------------

x_cpu = torch.tensor([1, 2, 3])
print()
print('x_cpu:', x_cpu)
print('x_cpu 所在设备:', x_cpu.device)

# ------------------------------------------------------------
# 3. 创建张量时直接指定设备
# ------------------------------------------------------------

# 直接在 device 上创建张量。
# 如果有 GPU，这个张量就在 GPU 上；如果没有 GPU，就在 CPU 上。
X = torch.randn(8, 3, device=device)
y = torch.randn(8, 1, device=device)

print()
print('X 所在设备:', X.device)
print('y 所在设备:', y.device)

# ------------------------------------------------------------
# 4. 使用 .to(device) 移动张量
# ------------------------------------------------------------

# 先在 CPU 上创建一个张量。
A = torch.ones(2, 3)

# 再移动到目标设备。
# 推荐初学者使用 .to(device)，因为它能兼容 CPU 和 GPU。
B = A.to(device)

print()
print('A 所在设备:', A.device)
print('B 所在设备:', B.device)

# 同一次计算中的张量必须在同一个设备上。
# B 和 C 都在 device 上，所以可以相加。
C = torch.zeros(2, 3, device=device)
D = B + C
print('D = B + C 的设备:', D.device)

# ------------------------------------------------------------
# 5. 把神经网络模型移动到同一个设备
# ------------------------------------------------------------

# 构造一个非常小的神经网络：输入 3 个特征，输出 1 个预测值。
net = nn.Sequential(
    nn.Linear(3, 4),
    nn.ReLU(),
    nn.Linear(4, 1)
)

# 将模型参数移动到 device。
# 注意：模型在 GPU，输入也必须在同一个 GPU；模型在 CPU，输入也应在 CPU。
net = net.to(device)

print()
print('第一层权重所在设备:', net[0].weight.device)
print('最后一层偏置所在设备:', net[2].bias.device)

# ------------------------------------------------------------
# 6. 在同一个设备上完成一次简单训练
# ------------------------------------------------------------

loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(net.parameters(), lr=0.03)

for epoch in range(10):
    # 清空上一轮梯度。
    optimizer.zero_grad()

    # 前向传播：X 和 net 都在同一个 device 上，所以可以正常计算。
    y_hat = net(X)

    # 计算损失：y_hat 和 y 也必须在同一个 device 上。
    loss = loss_fn(y_hat, y)

    # 反向传播。
    loss.backward()

    # 更新参数。
    optimizer.step()

    print(f'epoch {epoch + 1:2d}, loss={loss.item():.4f}, loss 所在设备={loss.device}')

# ------------------------------------------------------------
# 7. 如果需要把 GPU 上的结果拿回 CPU
# ------------------------------------------------------------

# 很多时候，打印、转 NumPy、保存结果时，会把张量移动回 CPU。
# 如果当前 device 本来就是 CPU，这句也仍然可以正常运行。
y_hat_cpu = y_hat.detach().to('cpu')

print()
print('y_hat 原始设备:', y_hat.device)
print('y_hat_cpu 设备:', y_hat_cpu.device)
print('前 3 个预测值:', y_hat_cpu[:3])

GPU 数量: 1
所有可用设备: [device(type='cuda', index=0)]
当前使用设备: cuda:0

x_cpu: tensor([1, 2, 3])
x_cpu 所在设备: cpu

X 所在设备: cuda:0
y 所在设备: cuda:0

A 所在设备: cpu
B 所在设备: cuda:0
D = B + C 的设备: cuda:0

第一层权重所在设备: cuda:0
最后一层偏置所在设备: cuda:0
epoch  1, loss=0.4593, loss 所在设备=cuda:0
epoch  2, loss=0.4119, loss 所在设备=cuda:0
epoch  3, loss=0.3711, loss 所在设备=cuda:0
epoch  4, loss=0.3362, loss 所在设备=cuda:0
epoch  5, loss=0.3064, loss 所在设备=cuda:0
epoch  6, loss=0.2810, loss 所在设备=cuda:0
epoch  7, loss=0.2593, loss 所在设备=cuda:0
epoch  8, loss=0.2410, loss 所在设备=cuda:0
epoch  9, loss=0.2254, loss 所在设备=cuda:0
epoch 10, loss=0.2123, loss 所在设备=cuda:0

y_hat 原始设备: cuda:0
y_hat_cpu 设备: cpu
前 3 个预测值: tensor([[-0.2926],
        [-0.2177],
        [-0.1726]])


# 1. 确认GPU

In [1]:
!nvidia-smi

Tue May 26 16:09:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4090        Off |   00000000:51:00.0 Off |                  Off |
| 46%   33C    P8             31W /  450W |   22488MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import torch
from torch import nn

torch.device('cpu'), torch.cuda.device('cuda'), torch.cuda.device('cuda:0')

(device(type='cpu'),
 <torch.cuda.device at 0x74edfdec33d0>)

In [3]:
print(torch.cuda.device_count())

1


In [10]:
def try_gpu(i=0):
    """如果存在，则返回gpu(i)，否则返回gpu"""
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')

def try_all_gpus():
    """返回所有可用的GPU，如果没有GPU，则返回[cpu(),]。"""
    devices = [torch.device(f'cuda:{i}') for i in range(torch.cuda.device_count())]      
    return devices if devices else [torch.device('cpu')]
                    
print(try_gpu())
print(try_gpu(10))
print(try_all_gpus())

cuda:0
cpu
[device(type='cuda', index=0), device(type='cuda', index=1), device(type='cuda', index=2), device(type='cuda', index=3)]


In [11]:
# 查询张量所在设备
X = torch.tensor([1,2,3])
print(X.device) # 默认在CPU内存上

cpu


In [12]:
# 存储在GPU上
X = torch.ones(2,3,device=try_gpu())
X 

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [13]:
# 在第二个GPU上创建一个随即张量
Y = torch.rand(2,3,device=try_gpu(1))
print(Y.device) # 没有1号GPU，则放到CPU上
Y

cuda:1


tensor([[0.6188, 0.0115, 0.6101],
        [0.3390, 0.1095, 0.1298]], device='cuda:1')

In [18]:
# 要计算X+Y，我们需要决定在哪里执行这个操作
Z = X.cuda(0) # X+Y必须X和Y都在同一个GPU上
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')


In [21]:
Y = torch.rand(2,3,device=try_gpu(0))
print(Y + Z)
Z.cuda(0) is Z # 如果变量在0号GPU时，就返回True

tensor([[1.8523, 1.3706, 1.9944],
        [1.3185, 1.9515, 1.8636]], device='cuda:0')


True

In [22]:
# 神经网络与GPU
net = nn.Sequential(nn.Linear(3,1)) # 创建神经网络时已经把权重初始化好了
net = net.to(device=try_gpu()) # 把所有参数在0号GPU上拷贝一份
X = torch.ones(2,3,device=try_gpu()) # X 在0号GPU上
net(X) # 所以前项运算所有元素都在0号GPU上运行

tensor([[0.9475],
        [0.9475]], device='cuda:0', grad_fn=<AddmmBackward0>)

In [23]:
# 确认模型参数存储在同一个GPU上
net[0].weight.data.device

device(type='cuda', index=0)